# Qwen3-1.7B 个人风格 LoRA（云端 GPU）

数据：Drive `clipvault-personal-lm/lora/{train,valid}.jsonl`（{"text": doc} 续写格式）。
产出：**merge 后的完整模型**（merge_and_unload）→ Drive → mac `mlx_lm.convert` 量化 4bit → `mlx_lm.server` 即成 Qwen 后端。

运行时：GPU（A100/L4 最佳；T4 也能跑，batch 调小）。

In [ ]:
# 1) 安装 unsloth（Colab 微调事实标准；HF TRL 亦可）
%pip install -q unsloth
!nvidia-smi -L

In [ ]:
# 2) 挂载 Drive
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/clipvault-personal-lm'
!ls -la {DATA_DIR}/lora/ 2>/dev/null || echo '需要 lora/{train,valid}.jsonl（ostore gdrive 同步或手动上传）'

In [ ]:
# 3) 加载 Qwen3-1.7B（4bit QLoRA，T4 16GB 也装得下）
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen3-1.7B',
    max_seq_length=1024,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
)
model.print_trainable_parameters()

In [ ]:
# 4) 数据：{"text": ...} 续写式（个人语料）
from datasets import load_dataset
ds = load_dataset('json', data_files={
    'train': f'{DATA_DIR}/lora/train.jsonl',
    'valid': f'{DATA_DIR}/lora/valid.jsonl',
})

from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds['train'],
    eval_dataset=ds['valid'],
    dataset_text_field='text',
    max_seq_length=1024,
    packing=True,   # 短文档打包，吞吐拉满
    args=SFTConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=2,           # 16.7M tokens ≈ 2 epochs
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        logging_steps=20,
        eval_strategy='steps', eval_steps=200,
        save_strategy='steps', save_steps=500,
        output_dir=f'{DATA_DIR}/qwen_lora_out',
        bf16=True,
        report_to='none',
    ),
)
trainer.train()

In [ ]:
# 5) 试写（验证个人风格已对齐）
FastLanguageModel.for_inference(model)
for prefix in ['## cf 迭代复盘\n- ', '写个脚本批量检查']:
    inputs = tokenizer(prefix, return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=64, do_sample=True,
                         temperature=0.8, top_p=0.9)
    print('PREFIX:', prefix.replace('\n', ' | '))
    print('OUT   :', tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)[:200])
    print()

In [ ]:
# 6) merge 并保存完整模型到 Drive（mac 端 mlx_lm.convert 直接可用）
merged_dir = f'{DATA_DIR}/merged/qwen3-1.7b-personal'
model.save_pretrained_merged(merged_dir, tokenizer, save_method='merged_16bit')
print('merged →', merged_dir)
!du -sh {merged_dir}

## 回到 mac 的收尾
```sh
# 1) ostore gdrive 同步回本地（或手动下载 merged 目录）
# 2) 转 mlx 4bit
personal-lm/.venv/bin/python -m mlx_lm convert \
  --hf-path ~/Library/Application\ Support/Keepsake/models/merged/qwen3-1.7b-personal \
  --mlx-path ~/Library/Application\ Support/Keepsake/models/qwen3-1.7b-personal-mlx -q
# 3) 服务
personal-lm/.venv/bin/python -m mlx_lm server \
  --model ~/Library/Application\ Support/Keepsake/models/qwen3-1.7b-personal-mlx --port 8790
# cv-lm :8788 的 /compose 与 /complete 自动用上（OpenAI 兼容后端不变）
```

隐私：语料与 merge 后的模型都含个人数据 —— Drive 是你信任的私有存储；不传 HF Hub、不入 git。